# COMPAS Fairness Audit - Practical Assignment

## Part 3: Hands-On Fairness Analysis

**Objective:** Conduct a comprehensive fairness audit of the COMPAS recidivism risk assessment tool to identify algorithmic bias and propose mitigation strategies.

### Background

COMPAS (Correctional Offender Management Profiling for Alternative Sanctions) is a risk assessment tool used in criminal justice to predict recidivism. A 2016 ProPublica investigation found significant racial bias in the system.

### What You'll Learn

- Load and explore sensitive criminal justice data
- Calculate fairness metrics (Disparate Impact, Equal Opportunity, Equalized Odds)
- Visualize bias across demographic groups
- Apply bias mitigation techniques using AIF360
- Write a professional audit report

---

## Setup: Import Required Libraries

First, let's import all necessary libraries. If you encounter import errors, run:

```bash
pip install numpy pandas matplotlib seaborn scikit-learn aif360
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing import AdversarialDebiasing

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

---

## Step 1: Load and Explore the COMPAS Dataset

### Dataset Information

The COMPAS dataset contains criminal justice data including:
- **Demographic information**: race, sex, age
- **Criminal history**: number of prior offenses
- **COMPAS scores**: risk assessment scores (1-10)
- **Outcomes**: whether the person re-offended within 2 years

### Download Instructions

If you haven't downloaded the dataset yet, run the cell below to fetch it automatically.

In [ ]:
import os
import urllib.request

DATA_PATH = '../data/compas_dataset.csv'

if not os.path.exists(DATA_PATH):
    print("📥 Downloading COMPAS dataset...")
    url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
    
    os.makedirs('../data', exist_ok=True)
    urllib.request.urlretrieve(url, DATA_PATH)
    print(f"✅ Dataset downloaded to {DATA_PATH}")
else:
    print(f"✅ Dataset already exists at {DATA_PATH}")

### Load and Inspect the Data

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumn names:")
print(df_raw.columns.tolist())

df_raw.head()

### Data Preprocessing

Following ProPublica's methodology, we'll filter the data to include only cases where:
- Screening date is within 30 days of arrest
- Recidivism information is available
- Charge degree is not 'Other'
- COMPAS score is available

In [ ]:
df = df_raw[
    (df_raw['days_b_screening_arrest'] <= 30) &
    (df_raw['days_b_screening_arrest'] >= -30) &
    (df_raw['is_recid'] != -1) &
    (df_raw['c_charge_degree'] != 'O') &
    (df_raw['score_text'] != 'N/A')
].copy()

print(f"Filtered dataset shape: {df.shape}")
print(f"Removed {len(df_raw) - len(df)} records during preprocessing")

### Exploratory Data Analysis

In [ ]:
print("📊 DEMOGRAPHIC DISTRIBUTION\n")

print("Race distribution:")
print(df['race'].value_counts())
print(f"\nPercentage:")
print(df['race'].value_counts(normalize=True) * 100)

print("\n" + "="*50)
print("\nGender distribution:")
print(df['sex'].value_counts())
print(f"\nPercentage:")
print(df['sex'].value_counts(normalize=True) * 100)

print("\n" + "="*50)
print("\nRecidivism rate:")
print(df['two_year_recid'].value_counts())
print(f"\nOverall recidivism rate: {df['two_year_recid'].mean():.2%}")

### Visualize Demographic Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['race'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution by Race', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Race')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

df['sex'].value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Distribution by Gender', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

df['age_cat'].value_counts().plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Distribution by Age Category', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Age Category')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---

## Step 2: Prepare Data for Fairness Analysis

We'll create binary variables for:
- **Protected attribute**: Race (African-American vs. Others)
- **Prediction**: High risk (COMPAS score >= 5)
- **Ground truth**: Actually re-offended within 2 years

In [ ]:
df['race_binary'] = df['race'].apply(lambda x: 1 if x == 'African-American' else 0)
df['sex_binary'] = df['sex'].apply(lambda x: 1 if x == 'Male' else 0)

df['high_risk'] = (df['decile_score'] >= 5).astype(int)

print("Binary variables created:")
print(f"  • race_binary: 1 = African-American, 0 = Other")
print(f"  • sex_binary: 1 = Male, 0 = Female")
print(f"  • high_risk: 1 = COMPAS score >= 5, 0 = COMPAS score < 5")

print(f"\nHigh risk classification rate: {df['high_risk'].mean():.2%}")
print(f"Actual recidivism rate: {df['two_year_recid'].mean():.2%}")

---

## Step 3: Calculate Fairness Metrics

### Key Fairness Metrics

1. **Disparate Impact**: Ratio of favorable outcomes between unprivileged and privileged groups
   - Fair range: 0.8 to 1.25
   
2. **Statistical Parity Difference**: Difference in positive prediction rates
   - Fair range: -0.1 to 0.1
   
3. **Equal Opportunity Difference**: Difference in True Positive Rates (TPR)
   - Fair range: -0.1 to 0.1
   
4. **Equalized Odds**: Both TPR and FPR should be equal across groups

In [ ]:
def calculate_fairness_metrics(df, protected_attr='race_binary', 
                               label_col='two_year_recid', 
                               pred_col='high_risk'):
    """
    Calculate comprehensive fairness metrics.
    """
    privileged = df[df[protected_attr] == 0]
    unprivileged = df[df[protected_attr] == 1]
    
    metrics = {}
    
    priv_positive_rate = privileged[pred_col].mean()
    unpriv_positive_rate = unprivileged[pred_col].mean()
    
    metrics['disparate_impact'] = unpriv_positive_rate / priv_positive_rate if priv_positive_rate > 0 else 0
    metrics['statistical_parity_diff'] = unpriv_positive_rate - priv_positive_rate
    
    def calc_rates(group_df):
        y_true = group_df[label_col]
        y_pred = group_df[pred_col]
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        
        return {
            'TPR': tp / (tp + fn) if (tp + fn) > 0 else 0,
            'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
            'TNR': tn / (tn + fp) if (tn + fp) > 0 else 0,
            'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0,
            'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn
        }
    
    priv_rates = calc_rates(privileged)
    unpriv_rates = calc_rates(unprivileged)
    
    metrics['privileged_TPR'] = priv_rates['TPR']
    metrics['unprivileged_TPR'] = unpriv_rates['TPR']
    metrics['equal_opportunity_diff'] = unpriv_rates['TPR'] - priv_rates['TPR']
    
    metrics['privileged_FPR'] = priv_rates['FPR']
    metrics['unprivileged_FPR'] = unpriv_rates['FPR']
    metrics['equalized_odds_diff'] = abs(unpriv_rates['TPR'] - priv_rates['TPR']) + abs(unpriv_rates['FPR'] - priv_rates['FPR'])
    
    metrics['privileged_rates'] = priv_rates
    metrics['unprivileged_rates'] = unpriv_rates
    
    return metrics

metrics = calculate_fairness_metrics(df)
print("✅ Fairness metrics calculated successfully!")

### Display Fairness Metrics Report

In [ ]:
print("\n" + "="*70)
print("FAIRNESS AUDIT REPORT - COMPAS RECIDIVISM PREDICTION")
print("="*70)

print(f"\n📊 DATASET STATISTICS")
print(f"   Total samples: {len(df)}")
print(f"   Privileged group (Non-Black): {(df['race_binary'] == 0).sum()}")
print(f"   Unprivileged group (Black): {(df['race_binary'] == 1).sum()}")

print(f"\n⚖️  FAIRNESS METRICS")

print(f"\n1. Disparate Impact: {metrics['disparate_impact']:.3f}")
if 0.8 <= metrics['disparate_impact'] <= 1.25:
    print("   ✅ PASS - Within acceptable range (0.8-1.25)")
else:
    print("   ❌ FAIL - Outside acceptable range (0.8-1.25)")
    print(f"   ⚠️  Interpretation: Unprivileged group is {metrics['disparate_impact']:.2f}x as likely to be classified as high risk")

print(f"\n2. Statistical Parity Difference: {metrics['statistical_parity_diff']:.3f}")
if abs(metrics['statistical_parity_diff']) < 0.1:
    print("   ✅ PASS - Minimal difference")
else:
    print("   ❌ FAIL - Significant difference")
    print(f"   ⚠️  Interpretation: {abs(metrics['statistical_parity_diff']):.1%} difference in high-risk classification rates")

print(f"\n3. Equal Opportunity Difference (TPR): {metrics['equal_opportunity_diff']:.3f}")
if abs(metrics['equal_opportunity_diff']) < 0.1:
    print("   ✅ PASS - Minimal difference")
else:
    print("   ❌ FAIL - Significant difference")
    print(f"   ⚠️  Interpretation: Different accuracy for identifying actual recidivists across groups")

print(f"\n4. Equalized Odds Difference: {metrics['equalized_odds_diff']:.3f}")
if metrics['equalized_odds_diff'] < 0.1:
    print("   ✅ PASS - Minimal difference")
else:
    print("   ❌ FAIL - Significant difference")
    print(f"   ⚠️  Interpretation: Combined TPR and FPR differences indicate bias")

print(f"\n📈 DETAILED PERFORMANCE METRICS")
print(f"\n   Privileged Group (Non-Black):")
print(f"      True Positive Rate (Sensitivity): {metrics['privileged_TPR']:.3f}")
print(f"      False Positive Rate: {metrics['privileged_FPR']:.3f}")

print(f"\n   Unprivileged Group (Black):")
print(f"      True Positive Rate (Sensitivity): {metrics['unprivileged_TPR']:.3f}")
print(f"      False Positive Rate: {metrics['unprivileged_FPR']:.3f}")

print("\n" + "="*70)

---

## Step 4: Visualize Fairness Analysis

Let's create comprehensive visualizations to understand the bias patterns.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('COMPAS Fairness Analysis - Comprehensive Visualization', 
             fontsize=16, fontweight='bold', y=1.00)

race_groups = df.groupby('race_binary')['high_risk'].mean()
race_labels = ['Non-Black\n(Privileged)', 'Black\n(Unprivileged)']
bars1 = axes[0, 0].bar(race_labels, race_groups.values, 
                       color=['#3498db', '#e74c3c'], alpha=0.7, edgecolor='black', linewidth=2)
axes[0, 0].set_ylabel('High Risk Classification Rate', fontweight='bold', fontsize=11)
axes[0, 0].set_title('Prediction Rates by Race', fontweight='bold', fontsize=12)
axes[0, 0].set_ylim(0, max(race_groups.values) * 1.2)
axes[0, 0].grid(axis='y', alpha=0.3)

for bar in bars1:
    height = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=11)

x_pos = np.arange(2)
width = 0.35
tpr_values = [metrics['privileged_TPR'], metrics['unprivileged_TPR']]
fpr_values = [metrics['privileged_FPR'], metrics['unprivileged_FPR']]

bars2 = axes[0, 1].bar(x_pos - width/2, tpr_values, width, 
                       label='True Positive Rate', color='#2ecc71', alpha=0.7, edgecolor='black')
bars3 = axes[0, 1].bar(x_pos + width/2, fpr_values, width,
                       label='False Positive Rate', color='#e67e22', alpha=0.7, edgecolor='black')
axes[0, 1].set_ylabel('Rate', fontweight='bold', fontsize=11)
axes[0, 1].set_title('True Positive Rate vs False Positive Rate', fontweight='bold', fontsize=12)
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(race_labels)
axes[0, 1].legend(loc='upper right')
axes[0, 1].grid(axis='y', alpha=0.3)

for bars in [bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.2f}', ha='center', va='bottom', fontsize=9)

fairness_names = ['Disparate\nImpact', 'Stat. Parity\nDiff', 'Equal Opp.\nDiff', 'Equal. Odds\nDiff']
fairness_vals = [
    metrics['disparate_impact'],
    metrics['statistical_parity_diff'],
    metrics['equal_opportunity_diff'],
    metrics['equalized_odds_diff']
]

def is_fair(name, val):
    if 'Disparate' in name:
        return 0.8 <= val <= 1.25
    else:
        return abs(val) < 0.1

colors_fair = ['#2ecc71' if is_fair(n, v) else '#e74c3c' 
               for n, v in zip(fairness_names, fairness_vals)]

bars4 = axes[1, 0].bar(fairness_names, fairness_vals, 
                       color=colors_fair, alpha=0.7, edgecolor='black', linewidth=2)
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=1.5)
axes[1, 0].axhline(y=0.8, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1, 0].axhline(y=1.25, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1, 0].axhline(y=-0.1, color='purple', linestyle=':', linewidth=1.5, alpha=0.7)
axes[1, 0].axhline(y=0.1, color='purple', linestyle=':', linewidth=1.5, alpha=0.7)
axes[1, 0].set_ylabel('Metric Value', fontweight='bold', fontsize=11)
axes[1, 0].set_title('Fairness Metrics Summary', fontweight='bold', fontsize=12)
axes[1, 0].grid(axis='y', alpha=0.3)

for bar, val in zip(bars4, fairness_vals):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.2f}', ha='center', va='bottom' if height > 0 else 'top', 
                   fontweight='bold', fontsize=10)

confusion_data = pd.DataFrame({
    'Group': ['Privileged', 'Privileged', 'Unprivileged', 'Unprivileged'],
    'Metric': ['False Positive Rate', 'False Negative Rate', 'False Positive Rate', 'False Negative Rate'],
    'Value': [
        metrics['privileged_FPR'],
        metrics['privileged_rates']['FNR'],
        metrics['unprivileged_FPR'],
        metrics['unprivileged_rates']['FNR']
    ]
})

pivot_data = confusion_data.pivot(index='Metric', columns='Group', values='Value')
pivot_data.plot(kind='bar', ax=axes[1, 1], color=['#3498db', '#e74c3c'], alpha=0.7, edgecolor='black')
axes[1, 1].set_ylabel('Rate', fontweight='bold', fontsize=11)
axes[1, 1].set_title('Error Rates Comparison', fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel('')
axes[1, 1].legend(title='Group', title_fontsize=10)
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/compas_fairness_visualizations.png', dpi=300, bbox_inches='tight')
print("\n✅ Visualization saved to: ../outputs/compas_fairness_visualizations.png")
plt.show()

---

## Step 5: Analyze Specific Bias Patterns

Let's dig deeper into the ProPublica findings about false positives and false negatives.

In [ ]:
def analyze_errors_by_race(df):
    """
    Analyze false positives and false negatives by race.
    """
    results = []
    
    for race in ['African-American', 'Caucasian']:
        race_df = df[df['race'] == race]
        
        fp = ((race_df['high_risk'] == 1) & (race_df['two_year_recid'] == 0)).sum()
        fn = ((race_df['high_risk'] == 0) & (race_df['two_year_recid'] == 1)).sum()
        tp = ((race_df['high_risk'] == 1) & (race_df['two_year_recid'] == 1)).sum()
        tn = ((race_df['high_risk'] == 0) & (race_df['two_year_recid'] == 0)).sum()
        
        total = len(race_df)
        
        results.append({
            'Race': race,
            'Total': total,
            'False Positives': fp,
            'False Negatives': fn,
            'True Positives': tp,
            'True Negatives': tn,
            'FP Rate': fp / (fp + tn) if (fp + tn) > 0 else 0,
            'FN Rate': fn / (fn + tp) if (fn + tp) > 0 else 0
        })
    
    return pd.DataFrame(results)

error_analysis = analyze_errors_by_race(df)

print("\n" + "="*80)
print("DETAILED ERROR ANALYSIS BY RACE")
print("="*80)
print(error_analysis.to_string(index=False))
print("\n" + "="*80)

print("\n🔍 KEY FINDINGS:")
black_fp_rate = error_analysis[error_analysis['Race'] == 'African-American']['FP Rate'].values[0]
white_fp_rate = error_analysis[error_analysis['Race'] == 'Caucasian']['FP Rate'].values[0]
print(f"   • Black defendants have {black_fp_rate:.1%} false positive rate")
print(f"   • White defendants have {white_fp_rate:.1%} false positive rate")
print(f"   • Black defendants are {black_fp_rate/white_fp_rate:.2f}x more likely to be incorrectly labeled high risk")

black_fn_rate = error_analysis[error_analysis['Race'] == 'African-American']['FN Rate'].values[0]
white_fn_rate = error_analysis[error_analysis['Race'] == 'Caucasian']['FN Rate'].values[0]
print(f"\n   • White defendants have {white_fn_rate:.1%} false negative rate")
print(f"   • Black defendants have {black_fn_rate:.1%} false negative rate")
print(f"   • White defendants are {white_fn_rate/black_fn_rate:.2f}x more likely to be incorrectly labeled low risk")

---

## Step 6: Bias Mitigation with AIF360

Now let's apply bias mitigation techniques to see if we can improve fairness.

### Technique: Reweighing (Pre-processing)

Reweighing assigns different weights to training examples to ensure fairness.

In [ ]:
feature_cols = ['age', 'priors_count', 'juv_fel_count', 'juv_misd_count']
label_col = 'two_year_recid'
protected_attr = 'race_binary'

df_model = df[feature_cols + [label_col, protected_attr]].dropna()

dataset_orig = BinaryLabelDataset(
    df=df_model,
    label_names=[label_col],
    protected_attribute_names=[protected_attr],
    favorable_label=0,
    unfavorable_label=1
)

privileged_groups = [{protected_attr: 0}]
unprivileged_groups = [{protected_attr: 1}]

metric_orig = BinaryLabelDatasetMetric(
    dataset_orig,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print("\n📊 BEFORE BIAS MITIGATION:")
print(f"   Disparate Impact: {metric_orig.disparate_impact():.3f}")
print(f"   Statistical Parity Difference: {metric_orig.statistical_parity_difference():.3f}")

RW = Reweighing(unprivileged_groups=unprivileged_groups,
                privileged_groups=privileged_groups)
dataset_transf = RW.fit_transform(dataset_orig)

metric_transf = BinaryLabelDatasetMetric(
    dataset_transf,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print("\n📊 AFTER REWEIGHING:")
print(f"   Disparate Impact: {metric_transf.disparate_impact():.3f}")
print(f"   Statistical Parity Difference: {metric_transf.statistical_parity_difference():.3f}")

print("\n✅ Bias mitigation completed!")
print(f"\n💡 INTERPRETATION:")
print(f"   Disparate impact improved from {metric_orig.disparate_impact():.3f} to {metric_transf.disparate_impact():.3f}")
if 0.8 <= metric_transf.disparate_impact() <= 1.25:
    print(f"   ✅ Now within acceptable fairness range!")
else:
    print(f"   ⚠️  Still outside acceptable range - additional mitigation needed")

---

## Step 7: Write Your Audit Report (300+ words)

### Instructions

Based on your analysis above, write a comprehensive audit report that includes:

1. **Summary of Findings**: What bias did you discover?
2. **Quantitative Evidence**: Cite specific metrics from your analysis
3. **Impact Assessment**: Who is affected and how?
4. **Recommendations**: What should be done to address the bias?

### Template

---

## FAIRNESS AUDIT REPORT - COMPAS RECIDIVISM PREDICTION SYSTEM

**Auditor:** [Your Name]  
**Date:** [Current Date]  
**System Evaluated:** COMPAS Recidivism Risk Assessment Tool  
**Protected Attribute Analyzed:** Race (African-American vs. Non-African-American)

---

### Executive Summary

[Write your summary of key findings here - 50-75 words]

---

### Quantitative Findings

#### 1. Disparate Impact

[Discuss the disparate impact metric and what it reveals - include the actual value you calculated]

#### 2. Error Rate Disparities

[Discuss false positive and false negative rates across racial groups]

#### 3. Equal Opportunity and Equalized Odds

[Discuss differences in TPR and FPR between groups]

---

### Impact Assessment

[Describe the real-world impact of this bias on individuals and society - 75-100 words]

---

### Recommendations

#### Immediate Actions
1. [Recommendation 1]
2. [Recommendation 2]

#### Long-term Improvements
1. [Recommendation 3]
2. [Recommendation 4]

---

### Conclusion

[Final thoughts on the ethical implications - 50 words]

---

**Word Count:** [Your word count - must be 300+]

---

---

## Step 8: Additional Analysis (Optional)

### Gender-Based Fairness Analysis

Let's also examine fairness across gender groups.

In [ ]:
gender_metrics = calculate_fairness_metrics(df, protected_attr='sex_binary')

print("\n" + "="*70)
print("GENDER-BASED FAIRNESS ANALYSIS")
print("="*70)

print(f"\nDisparate Impact: {gender_metrics['disparate_impact']:.3f}")
if 0.8 <= gender_metrics['disparate_impact'] <= 1.25:
    print("✅ PASS - Gender fairness acceptable")
else:
    print("❌ FAIL - Gender bias detected")

print(f"\nEqual Opportunity Difference: {gender_metrics['equal_opportunity_diff']:.3f}")
print(f"\nMale TPR: {gender_metrics['unprivileged_TPR']:.3f}")
print(f"Female TPR: {gender_metrics['privileged_TPR']:.3f}")

print("\n" + "="*70)

---

## Step 9: Reflection Questions

Consider these questions as you complete your assignment:

1. **Fairness vs. Accuracy Trade-off**: Did bias mitigation improve fairness? Did it affect overall accuracy?

2. **Real-World Impact**: How might false positives and false negatives affect individuals in the criminal justice system differently?

3. **Ethical Considerations**: Should risk assessment tools be used at all in criminal justice? What are the alternatives?

4. **Your Responsibility**: As a future AI practitioner, what steps will you take to ensure fairness in your work?

---

## Submission Checklist

Before submitting, ensure you have:

- [ ] Completed all code cells and verified they run without errors
- [ ] Generated all required visualizations
- [ ] Written your audit report (300+ words)
- [ ] Included quantitative evidence from your analysis
- [ ] Provided actionable recommendations
- [ ] Saved this notebook with outputs visible
- [ ] Exported visualizations to the outputs folder

---

## Resources for Further Learning

- **ProPublica COMPAS Analysis**: https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing
- **AIF360 Documentation**: http://aif360.mybluemix.net/
- **Fairness Definitions Explained**: https://fairware.cs.umass.edu/papers/Verma.pdf
- **Criminal Justice Risk Assessment**: https://www.ncsc.org/~/media/Files/PDF/Topics/Gender%20and%20Racial%20Fairness/IB_Pretrial_Risk_Assessment_020516.ashx

---

**Congratulations on completing the COMPAS Fairness Audit!** 🎉

You've taken an important step in understanding how to identify and address bias in AI systems. This knowledge is crucial for building more equitable technology.